In [1]:
# VFL SHAP - Multi-Class Version
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
import shap
# -----------------------------

In [2]:
# 1. Load Dataset & Extract Multi-Class Labels
# -----------------------------
df = pd.read_csv("D:\\Projects\\Datasets\\undersampled_CIC2017_dataset.csv")
df = df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")

# Print dataset information
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")


# Extract and simplify label information
label_col = "label"

# Group similar attack types
def simplify_label(label_str):
    """Group similar attack types into simplified categories"""
    label_upper = label_str.upper()
    
    # DDoS (Distributed Denial of Service) -> DDOS
    if 'DDOS' in label_upper:
        return "DDOS"
    
    # All DoS variants (DoS GoldenEye, DoS Hulk, DoS Slowhttptest, DoS slowloris) -> DOS
    if 'DOS' in label_upper and 'DDOS' not in label_upper:
        return "DOS"
    
    # All Web Attack types -> WEBATTACK
    if 'WEB ATTACK' in label_upper or 'WEBATTACK' in label_upper:
        return "WEBATTACK"
    
    # Keep other labels as-is (simplified)
    if 'BENIGN' in label_upper:
        return "BENIGN"
    elif 'BOT' in label_upper:
        return "BOT"
    elif 'PORTSCAN' in label_upper or 'PORT SCAN' in label_upper:
        return "PORTSCAN"
    elif 'FTP' in label_upper and 'PATATOR' in label_upper:
        return "FTPPATATOR"
    elif 'SSH' in label_upper and 'PATATOR' in label_upper:
        return "SSHPATATOR"
    elif 'HEARTBLEED' in label_upper:
        return "HEARTBLEED"
    elif 'INFILTRATION' in label_upper:
        return "INFILTRATION"
    else:
        # Default: take first word or first 8 chars
        words = label_str.split()
        if len(words) > 0:
            return words[0].upper()[:8]
        return label_str.upper()[:8]

# Apply simplification
df['label_simplified'] = df[label_col].apply(simplify_label)

# Group classes with less than 200 rows into "OTHERS"
label_counts = df['label_simplified'].value_counts()
min_rows = 200

# Find labels with less than min_rows
small_labels = label_counts[label_counts < min_rows].index.tolist()

if len(small_labels) > 0:
    print(f"Grouping {len(small_labels)} classes with < {min_rows} rows into 'OTHERS':")
    for small_label in small_labels:
        count = label_counts[small_label]
        print(f"  {small_label}: {count} rows -> OTHERS")
    
    # Replace small labels with "OTHERS"
    df.loc[df['label_simplified'].isin(small_labels), 'label_simplified'] = 'OTHERS'

# Create mapping to numeric
unique_labels = sorted(df['label_simplified'].unique())
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
label_names_short = {idx: label for label, idx in label_mapping.items()}

df['label_numeric'] = df['label_simplified'].map(label_mapping)
label_col = 'label_numeric'
num_classes = len(unique_labels)

print(f"Found {num_classes} simplified label types:")
for orig_label in sorted(df['label_simplified'].unique()):
    num = label_mapping[orig_label]
    print(f"  {orig_label} -> {num}")

label_mapping_dict = label_names_short

print(f"\nNumber of classes: {num_classes}")
print(f"Label distribution:\n{df[label_col].value_counts().sort_index()}")
# -----------------------------

Total rows: 390398
Total columns: 89
Grouping 2 classes with < 200 rows into 'OTHERS':
  INFILTRATION: 122 rows -> OTHERS
  HEARTBLEED: 70 rows -> OTHERS
Found 9 simplified label types:
  BENIGN -> 0
  BOT -> 1
  DDOS -> 2
  DOS -> 3
  FTPPATATOR -> 4
  OTHERS -> 5
  PORTSCAN -> 6
  SSHPATATOR -> 7
  WEBATTACK -> 8

Number of classes: 9
Label distribution:
label_numeric
0    300332
1      1228
2     12582
3     44020
4      3974
5       192
6     20525
7      2978
8      4567
Name: count, dtype: int64


In [3]:
# 2. Semantic Vertical Feature Partition
# -----------------------------
non_feature_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "label", "label_numeric", "label_simplified"]
all_features = [col for col in df.columns if col not in non_feature_cols]

# Filter to only numeric columns (exclude string/object columns)
all_features = [col for col in all_features if df[col].dtype in ['int64', 'float64', 'int32', 'float32']]

print(f"Total features: {len(all_features)}")

# Group features by semantic meaning for better partitioning
def categorize_feature(feature_name):
    """Categorize feature into semantic groups"""
    feat_lower = feature_name.lower()
    
    # Packet statistics
    if any(x in feat_lower for x in ['packet', 'pkt', 'ps']):
        if any(x in feat_lower for x in ['src2dst', 'forward', 'fwd']):
            return 'packet_forward'
        elif any(x in feat_lower for x in ['dst2src', 'backward', 'bwd']):
            return 'packet_backward'
        elif any(x in feat_lower for x in ['bidirectional', 'bi']):
            return 'packet_bidirectional'
        else:
            return 'packet_general'
    
    # Timing/Interval features
    elif any(x in feat_lower for x in ['piat', 'iat', 'interval', 'time', 'duration']):
        if any(x in feat_lower for x in ['src2dst', 'forward', 'fwd']):
            return 'timing_forward'
        elif any(x in feat_lower for x in ['dst2src', 'backward', 'bwd']):
            return 'timing_backward'
        elif any(x in feat_lower for x in ['bidirectional', 'bi']):
            return 'timing_bidirectional'
        else:
            return 'timing_general'
    
    # Protocol/Port features
    elif any(x in feat_lower for x in ['udp', 'tcp', 'port', 'protocol', 'syn', 'ack', 'fin', 'rst', 'urg', 'cwr']):
        return 'protocol'
    
    # Size/Length features
    elif any(x in feat_lower for x in ['length', 'size', 'bytes', 'byte']):
        return 'size'
    
    # Statistical features
    elif any(x in feat_lower for x in ['mean', 'std', 'stddev', 'min', 'max', 'var']):
        return 'statistics'
    
    # Flow features
    elif any(x in feat_lower for x in ['flow', 'count', 'num']):
        return 'flow'
    
    else:
        return 'other'

# Categorize all features
feature_categories = {}
for feat in all_features:
    cat = categorize_feature(feat)
    if cat not in feature_categories:
        feature_categories[cat] = []
    feature_categories[cat].append(feat)

print(f"\nFeature categories: {list(feature_categories.keys())}")
for cat, feats in feature_categories.items():
    print(f"  {cat}: {len(feats)} features")

# Distribute categories to parties for diversity
num_parties = 3
party_features_list = [[] for _ in range(num_parties)]

# Shuffle categories for random distribution
categories_list = list(feature_categories.keys())
np.random.seed(42)
np.random.shuffle(categories_list)

# Distribute categories round-robin style
for idx, cat in enumerate(categories_list):
    party_idx = idx % num_parties
    party_features_list[party_idx].extend(feature_categories[cat])

# Shuffle features within each party for randomness
for i in range(num_parties):
    np.random.shuffle(party_features_list[i])

party1_features = party_features_list[0]
party2_features = party_features_list[1]
party3_features = party_features_list[2]

print(f"\nParty 1: {len(party1_features)} features")
print(f"Party 2: {len(party2_features)} features")
print(f"Party 3: {len(party3_features)} features")

# Generate party names, domains, and actions based on feature categories
def analyze_feature_categories(features):
    """Analyze which categories are dominant in the feature set"""
    categories = {}
    for feat in features:
        cat = categorize_feature(feat)
        categories[cat] = categories.get(cat, 0) + 1
    return categories

def generate_party_name(features, party_num):
    categories = analyze_feature_categories(features)
    dominant_cat = max(categories.items(), key=lambda x: x[1])[0] if categories else 'other'
    
    name_map = {
        'packet_forward': f"Forward_Packet_Analysis_Party{party_num}",
        'packet_backward': f"Backward_Packet_Analysis_Party{party_num}",
        'packet_bidirectional': f"Bidirectional_Packet_Analysis_Party{party_num}",
        'packet_general': f"Packet_Statistics_Party{party_num}",
        'timing_forward': f"Forward_Timing_Analysis_Party{party_num}",
        'timing_backward': f"Backward_Timing_Analysis_Party{party_num}",
        'timing_bidirectional': f"Bidirectional_Timing_Analysis_Party{party_num}",
        'timing_general': f"Timing_Statistics_Party{party_num}",
        'protocol': f"Protocol_Analysis_Party{party_num}",
        'size': f"Packet_Size_Analysis_Party{party_num}",
        'statistics': f"Statistical_Analysis_Party{party_num}",
        'flow': f"Flow_Analysis_Party{party_num}",
        'other': f"Network_Analysis_Party{party_num}"
    }
    return name_map.get(dominant_cat, f"Network_Party{party_num}")

def generate_domain(features, party_num):
    categories = analyze_feature_categories(features)
    dominant_cat = max(categories.items(), key=lambda x: x[1])[0] if categories else 'other'
    
    domain_map = {
        'packet_forward': "Forward Packet Traffic Monitoring",
        'packet_backward': "Return Path Packet Analysis",
        'packet_bidirectional': "Bidirectional Packet Flow Analysis",
        'packet_general': "Packet Statistics & Analysis",
        'timing_forward': "Forward Path Timing Analysis",
        'timing_backward': "Return Path Timing Analysis",
        'timing_bidirectional': "Bidirectional Timing Analysis",
        'timing_general': "Network Timing & Latency Analysis",
        'protocol': "Protocol & Port Analysis",
        'size': "Packet Size & Payload Analysis",
        'statistics': "Statistical Network Analysis",
        'flow': "Network Flow Analysis",
        'other': f"Network Segment {party_num} Analysis"
    }
    return domain_map.get(dominant_cat, f"Network Segment {party_num}")

def generate_action(features, party_num):
    categories = analyze_feature_categories(features)
    dominant_cat = max(categories.items(), key=lambda x: x[1])[0] if categories else 'other'
    
    action_map = {
        'packet_forward': "implement forward path packet filtering and rate limiting",
        'packet_backward': "monitor and restrict return path packet traffic",
        'packet_bidirectional': "apply bidirectional packet inspection and filtering",
        'packet_general': "configure packet-level security policies and filtering",
        'timing_forward': "adjust forward path connection timeouts and rate limits",
        'timing_backward': "monitor return path timing anomalies and apply throttling",
        'timing_bidirectional': "optimize bidirectional flow timeouts and connection limits",
        'timing_general': "adjust network timing parameters and connection management",
        'protocol': "implement protocol-specific filtering and port-based access control",
        'size': "enforce packet size restrictions and payload inspection",
        'statistics': "apply statistical anomaly detection and adaptive policies",
        'flow': "manage flow state tables and connection tracking policies",
        'other': f"apply comprehensive network security policies for party {party_num}"
    }
    return action_map.get(dominant_cat, f"apply network security policies for party {party_num}")

party_names = [
    generate_party_name(party1_features, 1),
    generate_party_name(party2_features, 2),
    generate_party_name(party3_features, 3)
]

party_feature_groups = [
    f"Features: {', '.join(party1_features[:3])}..." if len(party1_features) > 3 else f"Features: {', '.join(party1_features)}",
    f"Features: {', '.join(party2_features[:3])}..." if len(party2_features) > 3 else f"Features: {', '.join(party2_features)}",
    f"Features: {', '.join(party3_features[:3])}..." if len(party3_features) > 3 else f"Features: {', '.join(party3_features)}"
]

party_domains = [
    generate_domain(party1_features, 1),
    generate_domain(party2_features, 2),
    generate_domain(party3_features, 3)
]

party_actions = [
    generate_action(party1_features, 1),
    generate_action(party2_features, 2),
    generate_action(party3_features, 3)
]

print("\n=== Party Configuration ===")
for i in range(3):
    features_list = [party1_features, party2_features, party3_features][i]
    print(f"\n{party_names[i]}:")
    print(f"  Domain: {party_domains[i]}")
    print(f"  Features ({len(features_list)}): {features_list[:5]}")
    print(f"  Action: {party_actions[i]}")
# -----------------------------

Total features: 88

Feature categories: ['timing_bidirectional', 'packet_bidirectional', 'size', 'timing_forward', 'packet_forward', 'timing_backward', 'packet_backward', 'packet_general']
  timing_bidirectional: 5 features
  packet_bidirectional: 17 features
  size: 3 features
  timing_forward: 5 features
  packet_forward: 15 features
  timing_backward: 5 features
  packet_backward: 13 features
  packet_general: 25 features

Party 1: 47 features
Party 2: 21 features
Party 3: 20 features

=== Party Configuration ===

Packet_Statistics_Party1:
  Domain: Packet Statistics & Analysis
  Features (47): ['udps.udp_packet_count', 'src2dst_duration_ms', 'bidirectional_ece_packets', 'bidirectional_fin_packets', 'src2dst_mean_piat_ms']
  Action: configure packet-level security policies and filtering

Backward_Packet_Analysis_Party2:
  Domain: Return Path Packet Analysis
  Features (21): ['dst2src_syn_packets', 'dst2src_ece_packets', 'dst2src_max_piat_ms', 'dst2src_packets', 'dst2src_min_ps']
  A

In [4]:
# 3. Preprocess Features
# -----------------------------
scaler1 = StandardScaler()
scaler2 = StandardScaler()
scaler3 = StandardScaler()

X1 = torch.tensor(scaler1.fit_transform(df[party1_features].values), dtype=torch.float32)
X2 = torch.tensor(scaler2.fit_transform(df[party2_features].values), dtype=torch.float32)
X3 = torch.tensor(scaler3.fit_transform(df[party3_features].values), dtype=torch.float32)

# Ensure we use numeric labels
if 'label_numeric' in df.columns:
    y = torch.tensor(df['label_numeric'].values, dtype=torch.long)  # long for multi-class
else:
    # Fallback: convert label_col to numeric if needed
    if df[label_col].dtype == 'object' or df[label_col].dtype == 'string':
        # Convert string labels to numeric
        unique_labels = sorted(df[label_col].unique())
        label_to_num = {label: idx for idx, label in enumerate(unique_labels)}
        y = torch.tensor(df[label_col].map(label_to_num).values, dtype=torch.long)
    else:
        y = torch.tensor(df[label_col].values, dtype=torch.long)

x_parts = [X1, X2, X3]
# -----------------------------

In [5]:
# 4. Train/Test Split (Improved Stratification)
# -----------------------------
# Use numeric labels for stratification
stratify_labels = df['label_numeric'] if 'label_numeric' in df.columns else y.numpy()

# Check class distribution before split
print("Class distribution before split:")
print(stratify_labels.value_counts().sort_index())

train_idx, test_idx = train_test_split(
    range(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels
)

x_train_parts = [x[train_idx] for x in x_parts]
x_test_parts = [x[test_idx] for x in x_parts]
y_train = y[train_idx]
y_test = y[test_idx]

# Verify stratification
print("\nTrain set class distribution:")
train_labels = stratify_labels.iloc[train_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[train_idx]
print(pd.Series(train_labels).value_counts().sort_index())

print("\nTest set class distribution:")
test_labels = stratify_labels.iloc[test_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[test_idx]
print(pd.Series(test_labels).value_counts().sort_index())
# -----------------------------

Class distribution before split:
label_numeric
0    300332
1      1228
2     12582
3     44020
4      3974
5       192
6     20525
7      2978
8      4567
Name: count, dtype: int64

Train set class distribution:
label_numeric
0    240265
1       982
2     10066
3     35216
4      3179
5       154
6     16420
7      2382
8      3654
Name: count, dtype: int64

Test set class distribution:
label_numeric
0    60067
1      246
2     2516
3     8804
4      795
5       38
6     4105
7      596
8      913
Name: count, dtype: int64


In [6]:
# 5. Multi-Class VFL Model (Improved Architecture)
# -----------------------------
class LocalEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=64, hidden_dim=128):
        super().__init__()
        # Deeper encoder for better feature learning
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.net(x)


class ActiveClassifier(nn.Module):
    def __init__(self, embed_dim=64, num_classes=5, hidden_dim=128):
        super().__init__()
        # Deeper classifier
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, h):
        # Return logits (NO softmax) - CrossEntropyLoss applies softmax internally
        return self.net(h)


class VFLModel(nn.Module):
    def __init__(self, input_dims, embed_dim=64, num_classes=5, hidden_dim=128):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_classes = num_classes
        self.encoders = nn.ModuleList([LocalEncoder(dim, embed_dim, hidden_dim) for dim in input_dims])
        # Improved fusion: concat embeddings instead of sum
        fusion_dim = embed_dim * len(input_dims)  # 3 parties * embed_dim
        self.classifier = ActiveClassifier(fusion_dim, num_classes, hidden_dim)

    def forward(self, x_parts):
        embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]
        # Improved fusion: concatenate instead of sum for better separation
        h = torch.cat(embeddings, dim=1)  # [B, embed_dim*3] instead of [B, embed_dim]
        y_hat = self.classifier(h)
        return y_hat

    def get_party_embeddings(self, x_parts):
        self.eval()
        with torch.no_grad():
            embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]
        return embeddings
# -----------------------------

In [7]:
# 6. Train & Evaluate Functions (Multi-Class with Improvements)
# -----------------------------
def train_vfl(model, x_parts, y, epochs=100, lr=1e-3, use_class_weights=True):
    # Calculate class weights for imbalanced data
    if use_class_weights:
        y_np = y.cpu().numpy()
        class_counts = np.bincount(y_np)
        total = len(y_np)
        class_weights = total / (len(class_counts) * class_counts.astype(float))
        class_weights = torch.tensor(class_weights, dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f"Using class weights: {class_weights.numpy()}")
    else:
        criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    # Improved scheduler: lower patience, smaller threshold for plateau detection
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15, 
        threshold=0.0001, min_lr=1e-6
    )
    
    best_loss = float('inf')
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        y_hat = model(x_parts)
        loss = criterion(y_hat, y.long())
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step(loss)
        
        if epoch % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            improvement = best_loss - loss.item() if best_loss != float('inf') else 0
            print(f"[VFL] Epoch {epoch} | Loss: {loss.item():.4f} | LR: {current_lr:.6f} | Best: {best_loss:.4f} | Δ: {improvement:.4f}")
        
        if loss.item() < best_loss:
            improvement = best_loss - loss.item()
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

def evaluate(model, x_parts, y):
    model.eval()
    with torch.no_grad():
        y_hat = model(x_parts)
        y_pred = torch.argmax(y_hat, dim=1)  # Multi-class prediction
    acc = accuracy_score(y.cpu(), y_pred.cpu())
    rec = recall_score(y.cpu(), y_pred.cpu(), average='macro')
    f1 = f1_score(y.cpu(), y_pred.cpu(), average='macro')
    print(f"[VFL] Accuracy: {acc:.4f} | Recall: {rec:.4f} | F1-score: {f1:.4f}")
    return acc, rec, f1
# -----------------------------

In [ ]:
# 7. Train VFL Model
# -----------------------------
embed_dim = 64
hidden_dim = 128
model = VFLModel(
    input_dims=[len(party1_features), len(party2_features), len(party3_features)],
    embed_dim=embed_dim,
    num_classes=num_classes,
    hidden_dim=hidden_dim
)

print(f"Model parameters: embed_dim={embed_dim}, hidden_dim={hidden_dim}, num_classes={num_classes}")
print(f"Fusion: concat (embed_dim*3 = {embed_dim*3})")
# Use slightly higher LR for better convergence
train_vfl(model, x_train_parts, y_train, epochs=500, lr=0.002, use_class_weights=True)
acc, rec, f1 = evaluate(model, x_test_parts, y_test)

perf_df = pd.DataFrame({"Metric": ["Accuracy", "Recall", "F1"], "Value": [acc, rec, f1]})
perf_df.to_csv("vfl_shap_performance.csv", index=False)
# -----------------------------

Model parameters: embed_dim=64, hidden_dim=128, num_classes=9
Fusion: concat (embed_dim*3 = 192)
Using class weights: [1.4443219e-01 3.5338085e+01 3.4474468e+00 9.8540437e-01 1.0916011e+01
 2.2533766e+02 2.1133983e+00 1.4568430e+01 9.4969893e+00]


D:\Projects\AgenticAI\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1694: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  current = float(metrics)


[VFL] Epoch 0 | Loss: 2.1977 | LR: 0.002000 | Best: inf | Δ: 0.0000
[VFL] Epoch 10 | Loss: 1.1687 | LR: 0.002000 | Best: 1.2840 | Δ: 0.1153


In [ ]:
# 8. Build Party-Level Meta-Features
# -----------------------------
model.eval()
with torch.no_grad():
    train_embeds = model.get_party_embeddings(x_train_parts)
    test_embeds = model.get_party_embeddings(x_test_parts)

h1_train = train_embeds[0].mean(dim=1, keepdim=True)
h2_train = train_embeds[1].mean(dim=1, keepdim=True)
h3_train = train_embeds[2].mean(dim=1, keepdim=True)

h1_test = test_embeds[0].mean(dim=1, keepdim=True)
h2_test = test_embeds[1].mean(dim=1, keepdim=True)
h3_test = test_embeds[2].mean(dim=1, keepdim=True)

X_train_meta = torch.cat([h1_train, h2_train, h3_train], dim=1)
X_test_meta = torch.cat([h1_test, h2_test, h3_test], dim=1)
# -----------------------------

In [ ]:
# 9. Multi-Class Meta-Model
# -----------------------------
class PartyMetaModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.fc = nn.Linear(3, num_classes)
    def forward(self, x_meta):
        # Return logits (NO softmax) - CrossEntropyLoss applies softmax internally
        return self.fc(x_meta)

meta_model = PartyMetaModel(num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(meta_model.parameters(), lr=1e-3)
meta_epochs = 50

model.eval()
for epoch in range(meta_epochs):
    meta_model.train()
    optimizer.zero_grad()
    with torch.no_grad():
        y_hat_teacher = model(x_train_parts)
        y_teacher_pred = torch.argmax(y_hat_teacher, dim=1)  # Get class predictions
    y_hat_meta = meta_model(X_train_meta)
    loss = criterion(y_hat_meta, y_teacher_pred.long())
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"[META] Epoch {epoch} | Distillation Loss: {loss.item():.4f}")

# Meta-model fidelity
meta_model.eval()
with torch.no_grad():
    y_hat_teacher_test = model(x_test_parts)
    y_teacher_pred_test = torch.argmax(y_hat_teacher_test, dim=1).cpu().numpy()
    y_hat_meta_test = meta_model(X_test_meta)
    y_meta_pred_test = torch.argmax(y_hat_meta_test, dim=1).cpu().numpy()
meta_acc = accuracy_score(y_teacher_pred_test, y_meta_pred_test)
print(f"[META] Accuracy between VFL and meta-model predictions: {meta_acc:.4f}")
# -----------------------------

In [ ]:
# 10. SHAP on Party Meta-Features
# -----------------------------
meta_model.eval()

bg_size = min(100, X_train_meta.shape[0])
bg_idx = torch.randperm(X_train_meta.shape[0])[:bg_size]
background = X_train_meta[bg_idx].detach().cpu().numpy()

def meta_predict(x_np):
    x_t = torch.tensor(x_np, dtype=torch.float32)
    with torch.no_grad():
        logits = meta_model(x_t)
        # Apply softmax to get probabilities for SHAP
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

explainer = shap.KernelExplainer(meta_predict, background)

n_explain = min(200, X_test_meta.shape[0])
X_explain = X_test_meta[:n_explain].detach().cpu().numpy()

shap_values = explainer.shap_values(X_explain, nsamples=200)

# Get predictions for selecting correct SHAP values
y_meta_pred_explain = torch.argmax(meta_model(torch.tensor(X_explain, dtype=torch.float32)), dim=1).cpu().numpy()

# Handle SHAP output format - for multi-class, SHAP returns list of arrays
if isinstance(shap_values, list):
    # Multi-class: SHAP returns list where each element is [n_explain, 3] for that class
    # Select SHAP values for predicted class for each sample
    shap_values_selected = []
    for i in range(n_explain):
        pred_class = int(y_meta_pred_explain[i])
        if pred_class < len(shap_values):
            shap_values_selected.append(shap_values[pred_class][i])
        else:
            # Fallback: use first class if prediction is out of bounds
            shap_values_selected.append(shap_values[0][i])
    shap_values = np.array(shap_values_selected)  # [n_explain, 3]
else:
    shap_values = np.asarray(shap_values)
    print(f"SHAP values raw shape: {shap_values.shape}")
    
    if len(shap_values.shape) == 3:
        # Could be [num_classes, n_explain, 3] or [n_explain, num_classes, 3]
        if shap_values.shape[0] == num_classes and shap_values.shape[1] == n_explain:
            # Shape is [num_classes, n_explain, 3]
            shap_values_selected = []
            for i in range(n_explain):
                pred_class = int(y_meta_pred_explain[i])
                if pred_class < shap_values.shape[0]:
                    shap_values_selected.append(shap_values[pred_class, i, :])
                else:
                    shap_values_selected.append(shap_values[0, i, :])
            shap_values = np.array(shap_values_selected)  # [n_explain, 3]
        elif shap_values.shape[0] == n_explain and shap_values.shape[2] == num_classes:
            # Shape is [n_explain, 3, num_classes] - transpose needed
            shap_values_selected = []
            for i in range(n_explain):
                pred_class = int(y_meta_pred_explain[i])
                if pred_class < shap_values.shape[2]:
                    shap_values_selected.append(shap_values[i, :, pred_class])
                else:
                    shap_values_selected.append(shap_values[i, :, 0])
            shap_values = np.array(shap_values_selected)  # [n_explain, 3]
        else:
            # Try to extract based on predicted class
            print(f"Warning: Unexpected SHAP shape {shap_values.shape}, attempting to extract...")
            shap_values_selected = []
            for i in range(n_explain):
                pred_class = int(y_meta_pred_explain[i])
                # Try different indexing patterns
                try:
                    if shap_values.shape[0] == num_classes:
                        shap_values_selected.append(shap_values[pred_class, i, :])
                    else:
                        shap_values_selected.append(shap_values[i, :, pred_class])
                except:
                    shap_values_selected.append(shap_values[i, :, 0] if len(shap_values.shape) == 3 else shap_values[i, :])
            shap_values = np.array(shap_values_selected)

print(f"SHAP values final shape: {shap_values.shape}")
# -----------------------------

In [ ]:
# 11. Aggregate SHAP for Multi-Class
# -----------------------------
phi = shap_values  # [n_explain, 3]
phi_abs = np.abs(phi)

# Global mean |SHAP| and percentage per party
mean_phi_abs = phi_abs.mean(axis=0)
mean_phi_pct = mean_phi_abs / mean_phi_abs.sum()

print("\n=== Global SHAP Party Attributions ===")
global_rows = []
for i, name in enumerate(party_names):
    m_abs = float(mean_phi_abs[i])
    m_pct = float(mean_phi_pct[i])
    print(f"{name}: mean |SHAP| = {m_abs:.6f}, mean contribution = {m_pct*100:5.2f}%")
    global_rows.append({
        "Party": name,
        "Domain": party_domains[i],
        "Feature_Group": party_feature_groups[i],
        "Mean_abs_SHAP_All": m_abs,
        "Mean_contrib_All": m_pct,
    })
global_df = pd.DataFrame(global_rows)
global_df.to_csv("vfl_shap_global_summary.csv", index=False)

# Per-class SHAP analysis
y_test_np = y_test.cpu().numpy()
y_explain = y_test_np[:n_explain]

print("\n=== SHAP Party Attributions by Class ===")
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    print(f"\n{class_name} (class {class_idx}):")
    class_rows = []
    for i, name in enumerate(party_names):
        c_abs = float(mean_phi_class[i])
        c_pct = float(mean_pct_class[i])
        print(f"  {name}: mean |SHAP| = {c_abs:.6f}, mean contribution = {c_pct*100:5.2f}%")
        class_rows.append({
            "Party": name,
            "Domain": party_domains[i],
            "Feature_Group": party_feature_groups[i],
            "Class": class_name,
            "Class_Index": class_idx,
            "Mean_abs_SHAP": c_abs,
            "Mean_contrib": c_pct,
        })
    
    class_df = pd.DataFrame(class_rows)
    class_df.to_csv(f"vfl_shap_{class_name.lower()}_summary.csv", index=False)

# Find dominant party per class
print("\n=== Dominant Party per Class ===")
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    top_party_idx = int(np.argmax(mean_pct_class))
    top_party_name = party_names[top_party_idx]
    top_party_share = float(mean_pct_class[top_party_idx]) * 100.0
    
    print(f"{class_name}: {top_party_name} ({top_party_share:.2f}%)")
# -----------------------------

In [ ]:
# 12. Agent-Level Mitigation Summary
# -----------------------------
agent_rows = []
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    top_party_idx = int(np.argmax(mean_pct_class))
    
    for i, name in enumerate(party_names):
        agent_rows.append({
            "Class": class_name,
            "Class_Index": class_idx,
            "Party": name,
            "Domain": party_domains[i],
            "Feature_Group": party_feature_groups[i],
            "Mean_contrib": float(mean_pct_class[i]),
            "Suggested_Action": party_actions[i],
            "Is_Dominant": (i == top_party_idx)
        })

agent_df = pd.DataFrame(agent_rows)
agent_df.to_csv("vfl_shap_agent_mitigation_summary.csv", index=False)

print("\n=== Agentic Mitigation Recommendations ===")
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_agent = agent_df[(agent_df['Class'] == class_name) & (agent_df['Is_Dominant'] == True)]
    if len(class_agent) > 0:
        row = class_agent.iloc[0]
        print(f"\n{class_name}:")
        print(f"  Dominant Party: {row['Party']} ({row['Domain']})")
        print(f"  Contribution: {row['Mean_contrib']*100:.2f}%")
        print(f"  Recommended Action: {row['Suggested_Action']}")
# -----------------------------